<font color='rainbow' size=6pt> Supervised Machine Learning: Match Predictor

In [1]:
#set up environment
!python set_up.py

import os
pip_upgrade = os.path.join("venv", "Scripts", "python.exe") if os.name == "nt" else os.path.join("venv", "bin", "python")
!{pip_upgrade} -m pip install --upgrade pip

!pip install -r https://raw.githubusercontent.com/Keegan-McCullough/Soccer_Match_Predictor/main/requirements.txt

Installed dependencies
Installed kernelspec soccer-env in C:\Users\keega\AppData\Roaming\jupyter\kernels\soccer-env


In [2]:
import kagglehub
import pandas as pd
import os
import sqlite3

# Download latest version
epl_path = kagglehub.dataset_download("evangower/english-premier-league-standings")

epl_dfs = {}
# Load EPL CSVs into a dictionary of DataFrames
for f in os.listdir(epl_path):
    if f.endswith(".csv"):
        epl_dfs[f] = pd.read_csv(os.path.join(epl_path, f))


# Download Europe soccer SQLite database
europe_path = kagglehub.dataset_download("hugomathien/soccer")

# Find the file
db_file = [f for f in os.listdir(europe_path) if f.endswith(".sqlite")][0]
db_path = os.path.join(europe_path, db_file)

# Connect and list all tables
conn = sqlite3.connect(db_path)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
# Load each table into a dictionary of DataFrames
europe_dfs = {}
for table in tables["name"]:
    europe_dfs[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
conn.close()

c:\Users\keega\OneDrive\GitHub\Soccer_Match_Predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# EPL data structure overview
print("EPL datasets:", list(epl_dfs.keys()))
epl_rankings = epl_dfs['pl-tables-1993-2024.csv']
print(epl_rankings)

EPL datasets: ['pl-tables-1993-2024.csv']
     season_end_year               team  position  played  won  drawn  lost  \
0               1993     Manchester Utd         1      42   24     12     6   
1               1993        Aston Villa         2      42   21     11    10   
2               1993       Norwich City         3      42   21      9    12   
3               1993          Blackburn         4      42   20     11    11   
4               1993                QPR         5      42   17     12    13   
..               ...                ...       ...     ...  ...    ...   ...   
641             2024          Brentford        16      38   10      9    19   
642             2024  Nottingham Forest        17      38    9      9    20   
643             2024         Luton Town        18      38    6      8    24   
644             2024            Burnley        19      38    5      9    24   
645             2024      Sheffield Utd        20      38    3      7    28   

     gf  

In [4]:
# Europe DB tables overview
print("\nEurope DB tables:", tables["name"].tolist())

# Get all the data for only the englis teams
english_teams = ['Arsenal','Aston Villa', 'Blackburn Rovers', 'Bolton Wanderers', 
                 'Chelsea', 'Everton', 'Liverpool', 'Manchester City', 'Manchester United',
                 'Middlesbrough', 'Newcastle United', 'Queens Park Rangers', 'Southampton',
                 'Tottenham Hotspur', 'West Ham United', 'Burnley', 'Swansea', 'Wolverhampton Wanderers',
                 'Cardiff City', 'Hull City', 'Stoke City', 'Norwich City', 'West Bromwich Albion',
                 'Blackpool', 'Crystal Palace', 'Watford', 'Leicester City', 'Sunderland', 'Birmingham City',
                 'Fulham', 'Wigan Athletic', 'Swansea City', 'Reading', 'Bournemouth']

df_teams = europe_dfs["Team"]
team_map = {}

for index, team in df_teams.iterrows():
    if team['team_long_name'] in english_teams:
        team_map[team['team_api_id']] = team['team_long_name']

# map ids to name for readability 
def map_ids(df, team_map, name):
    df[name] = df[name].map(team_map)
    return df

# Collect only the data we want ie discrete values
df_attr = europe_dfs['Team_Attributes'][['team_api_id', 'date', 'buildUpPlaySpeed', 'buildUpPlayPassing',
                                         'chanceCreationPassing', 'chanceCreationCrossing', 'chanceCreationShooting',
                                         'defencePressure', 'defenceAggression', 'defenceTeamWidth' ]]

df_attr = df_attr[df_attr['team_api_id'].isin(team_map.keys())]
df_attr = map_ids(df_attr, team_map, 'team_api_id')

print(team_map)

df_match = europe_dfs['Match'][['league_id', 'season', 'date', 'home_team_api_id', 'away_team_api_id', 'home_team_goal',
                                'away_team_goal', 'B365H', 'B365D', 'B365A']]

df_match = map_ids(df_match, team_map, 'home_team_api_id')
df_match = map_ids(df_match, team_map, 'away_team_api_id')

epl = 1729
df_match = df_match[df_match['league_id'] == epl]
df_match = df_match[df_match['season'] != '2008/2009']
df_match = df_match[df_match['season'] != '2009/2010']

df_attr['date'] = pd.to_datetime(df_attr['date'])
df_match['date'] = pd.to_datetime(df_match['date'])

df_attr = df_attr.sort_values('date')
df_match = df_match.sort_values('date')

print(df_match.head())
#print(df_attr.head())


Europe DB tables: ['sqlite_sequence', 'Player_Attributes', 'Player', 'Match', 'League', 'Country', 'Team', 'Team_Attributes']
{10260: 'Manchester United', 10261: 'Newcastle United', 9825: 'Arsenal', 8659: 'West Bromwich Albion', 8472: 'Sunderland', 8650: 'Liverpool', 8654: 'West Ham United', 8528: 'Wigan Athletic', 10252: 'Aston Villa', 8456: 'Manchester City', 8668: 'Everton', 8655: 'Blackburn Rovers', 8549: 'Middlesbrough', 8586: 'Tottenham Hotspur', 8559: 'Bolton Wanderers', 10194: 'Stoke City', 8667: 'Hull City', 9879: 'Fulham', 8455: 'Chelsea', 8658: 'Birmingham City', 8602: 'Wolverhampton Wanderers', 8191: 'Burnley', 8483: 'Blackpool', 10003: 'Swansea City', 10172: 'Queens Park Rangers', 9850: 'Norwich City', 8466: 'Southampton', 9798: 'Reading', 9826: 'Crystal Palace', 8344: 'Cardiff City', 8197: 'Leicester City', 8678: 'Bournemouth', 9817: 'Watford'}
      league_id     season       date  home_team_api_id      away_team_api_id  \
2488       1729  2010/2011 2010-08-14       Ast

In [5]:
# Feature engineering and df connections

# Home team attributes
df_match = pd.merge_asof(
    df_match,
    df_attr,
    left_on='date',
    right_on='date',
    left_by='home_team_api_id',
    right_by='team_api_id',
    direction='backward'
)

df_match = df_match.rename(columns=lambda x: f"home_{x}" if x not in ['league_id','season','date','home_team_api_id','away_team_api_id','home_team_goal','away_team_goal','B365H','B365D','B365A'] else x)

# Repeat for away team
df_match = pd.merge_asof(
    df_match,
    df_attr,
    left_on='date',
    right_on='date',
    left_by='away_team_api_id',
    right_by='team_api_id',
    direction='backward'
)

df_match = df_match.rename(columns=lambda x: f"away_{x}" if 'home_' not in x and x not in ['league_id','season','date','home_team_api_id','away_team_api_id','home_team_goal','away_team_goal','B365H','B365D','B365A'] else x)

In [6]:
# Create team name -> team_api_id mapping

# Fix naming differences between datasets
name_fixes = {
    "Manchester Utd": "Manchester United",
    "Blackburn": "Blackburn Rovers",
    "QPR": "Queens Park Rangers",
    "Tottenham": "Tottenham Hotspur",
    "Newcastle Utd": "Newcastle United",
    "West Ham": "West Ham United",
    "Bolton": "Bolton Wanderers",
    "West Brom": "West Bromwich Albion",
    "Wolves": "Wolverhampton Wanderers",
    "Swansea" : "Swansea City",
    "Charlton Ath" : "Charlton Athletic"
}

# Clean whitespace
epl_rankings["team"] = epl_rankings["team"].str.strip()

# Apply name fixes
epl_rankings["team"] = epl_rankings["team"].replace(name_fixes)

In [7]:
df_match["season_end"] = df_match["season"].apply(lambda x: int(str(x).split("/")[-1]))
print(df_match[["season", "season_end"]])

         season  season_end
0     2010/2011        2011
1     2010/2011        2011
2     2010/2011        2011
3     2010/2011        2011
4     2010/2011        2011
...         ...         ...
2275  2015/2016        2016
2276  2015/2016        2016
2277  2015/2016        2016
2278  2015/2016        2016
2279  2015/2016        2016

[2280 rows x 2 columns]


In [8]:
points_lookup = epl_rankings.set_index(["team", "season_end_year"])["points"].to_dict()
print(epl_rankings[["team", "season_end_year", "points"]].head(10))

                  team  season_end_year  points
0    Manchester United             1993      84
1          Aston Villa             1993      74
2         Norwich City             1993      72
3     Blackburn Rovers             1993      71
4  Queens Park Rangers             1993      63
5            Liverpool             1993      59
6       Sheffield Weds             1993      59
7    Tottenham Hotspur             1993      59
8      Manchester City             1993      57
9              Arsenal             1993      56


In [9]:
import numpy as np

alpha = 0.5

weights = np.array([alpha**i for i in range(5)])
weights = weights / weights.sum()

print(weights)

[0.51612903 0.25806452 0.12903226 0.06451613 0.03225806]


In [10]:
def compute_team_rating(team, season_end, points_lookup, weights, fallback_points=20):
    
    # the 5 previous seasons
    years = [season_end - 1 - i for i in range(5)]
    
    # get points for those seasons (or fallback if missing)
    values = [points_lookup.get((team, y), fallback_points) for y in years]
    
    # compute weighted average
    rating = np.dot(weights, values)
    
    return rating

In [11]:
df_match = df_match.loc[:, ~df_match.columns.duplicated()]

In [12]:
all_teams = pd.unique(pd.concat([df_match["home_team_api_id"], df_match["away_team_api_id"]]))
all_seasons = sorted(df_match["season_end"].unique())

print(len(all_teams))
print(all_seasons)

32
[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]


In [13]:
rating_rows = []

for team in all_teams:
    for season in all_seasons:
        rating = compute_team_rating(team, season, points_lookup, weights)
        
        rating_rows.append({
            "team": team,
            "season_end": season,
            "rating": rating
        })

ratings_df = pd.DataFrame(rating_rows)

In [14]:
df_match = df_match.drop(
    columns=[col for col in df_match.columns if "rating" in col],
    errors="ignore"
)

df_match = df_match.merge(
    ratings_df.rename(columns={
        "team": "home_team_api_id",
        "rating": "home_rating"
    }),
    on=["home_team_api_id", "season_end"],
    how="left"
)

In [15]:
df_match = df_match.merge(
    ratings_df.rename(columns={
        "team": "away_team_api_id",
        "rating": "away_rating"
    }),
    on=["away_team_api_id", "season_end"],
    how="left"
)

In [33]:
df_match[[
    "home_team_api_id", "away_team_api_id", "season_end",
    "home_rating", "away_rating"
]].head(10)

,home_team_api_id,away_team_api_id,season_end,home_rating,away_rating
0,Aston Villa,West Ham United,2011,61.354839,41.967742
1,Blackburn Rovers,Everton,2011,49.258065,61.483871
2,Bolton Wanderers,Fulham,2011,40.903226,46.129032
3,Chelsea,West Bromwich Albion,2011,85.064516,23.419355
4,Sunderland,Birmingham City,2011,38.806452,37.870968
5,Tottenham Hotspur,Manchester City,2011,61.193548,58.677419
6,Wolverhampton Wanderers,Stoke City,2011,29.290323,40.387097
7,Wigan Athletic,Blackpool,2011,39.451613,20.000000
8,Liverpool,Arsenal,2011,71.548387,74.548387
9,Manchester United,Newcastle United,2011,86.741935,29.290323


In [34]:
# Create match outcome feature (home_win, away_win, draw)
df_model = df_match.copy()
df_model['match_outcome'] = df_model.apply(
    lambda row: 'home_win' if row['home_team_goal'] > row['away_team_goal'] 
                else ('away_win' if row['home_team_goal'] < row['away_team_goal'] 
                      else 'draw'), 
    axis=1
)

print("Match outcome distribution:")
print(df_model['match_outcome'].value_counts())
print(f"\nHome points: {(df_model['match_outcome'] == 'home_win').sum()}")
print(f"Away points: {(df_model['match_outcome'] == 'away_win').sum()}")
print(f"Draws: {(df_model['match_outcome'] == 'draw').sum()}")

Match outcome distribution:
match_outcome
home_win    1024
away_win     666
draw         590
Name: count, dtype: int64

Home points: 1024
Away points: 666
Draws: 590


In [35]:
df_model1 = df_model.copy()
df_model

,league_id,season,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,B365H,B365D,B365A,...,away_chanceCreationPassing,away_chanceCreationCrossing,away_chanceCreationShooting,away_defencePressure,away_defenceAggression,away_defenceTeamWidth,season_end,home_rating,away_rating,match_outcome
0,1729,2010/2011,2010-08-14,Aston Villa,West Ham United,3,0,2.00,3.30,4.00,...,31,70,50,30,70,30,2011,61.354839,41.967742,home_win
1,1729,2010/2011,2010-08-14,Blackburn Rovers,Everton,1,0,2.88,3.25,2.50,...,60,70,45,40,70,40,2011,49.258065,61.483871,home_win
2,1729,2010/2011,2010-08-14,Bolton Wanderers,Fulham,0,0,2.20,3.30,3.40,...,70,70,50,40,35,40,2011,40.903226,46.129032,draw
3,1729,2010/2011,2010-08-14,Chelsea,West Bromwich Albion,6,0,1.17,7.00,17.00,...,70,70,55,70,70,70,2011,85.064516,23.419355,home_win
4,1729,2010/2011,2010-08-14,Sunderland,Birmingham City,2,2,2.10,3.30,3.60,...,70,70,70,70,70,70,2011,38.806452,37.870968,draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2275,1729,2015/2016,2016-05-15,Chelsea,Leicester City,1,1,2.30,3.75,3.10,...,47,64,46,58,65,55,2016,82.161290,30.838710,draw
2276,1729,2015/2016,2016-05-15,Everton,Norwich City,3,0,1.80,4.00,4.50,...,46,54,47,38,39,51,2016,56.322581,28.193548,home_win
2277,1729,2015/2016,2016-05-15,Watford,Sunderland,2,2,2.05,3.75,3.70,...,59,50,55,47,45,49,2016,20.000000,38.870968,draw
2278,1729,2015/2016,2016-05-15,Arsenal,Aston Villa,4,0,1.17,9.00,17.00,...,60,48,38,35,44,54,2016,75.225806,38.709677,home_win


In [ ]:
# Prepare features for modeling
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Define columns to exclude from features
exclude_cols = [
    'league_id', 'season', 'date', 
    'home_team_api_id', 'away_team_api_id',
    'home_team_goal', 'away_team_goal',
    'season_end', 'match_outcome'  # Exclude match_outcome from features
]

# Get feature columns
feature_cols = [col for col in df_model.columns if col not in exclude_cols]

# Create active features list (excluding any that shouldn't be used)
active_features = [f for f in feature_cols if f != 'match_outcome']

# Create feature matrix and target variables
X = df_model[feature_cols]
y = df_model['home_team_goal']

print(f"Features: {len(feature_cols)}")
print(f"Active features: {len(active_features)}")
print(f"Samples: {len(X)}")
print(f"Missing values: {X.isnull().sum().sum()}")

In [ ]:
# Prepare away team goal target variable
y_away = df_model.loc[X.index, 'away_team_goal']

print(f"Away goals target variable prepared: {len(y_away)} samples")

In [ ]:
# Standardize features and split data for both home and away predictions
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split for home team goal prediction
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Split for away team goal prediction (using same random state for consistency)
X_train_away, X_test_away, y_train_away, y_test_away = train_test_split(
    X_scaled, y_away, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Model Training: Logistic Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix as cm

y_match = df_model['match_outcome']
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_scaled, y_match, test_size=0.2, random_state=42
)

logistic_lasso = LogisticRegression(
    penalty='l1',
    solver='saga',
    C=0.1,
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
)

logistic_lasso.fit(X_train_class, y_train_class)

train_accuracy = logistic_lasso.score(X_train_class, y_train_class)
test_accuracy = logistic_lasso.score(X_test_class, y_test_class)

print("Logistic Lasso Results:")
print(f"Training Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Classes: {list(logistic_lasso.classes_)}")

In [ ]:
# Model Evaluation
y_pred_train = logistic_lasso.predict(X_train_class)
y_pred_test = logistic_lasso.predict(X_test_class)

print("\nClassification Report (Test Set):")
print(classification_report(y_test_class, y_pred_test))

logistic_coef = logistic_lasso.coef_
print(f"Model shape: {logistic_coef.shape} ({len(logistic_lasso.classes_)} classes × {len(active_features)} features)")

In [ ]:
# Feature Importance Analysis
avg_coef = np.mean(np.abs(logistic_coef), axis=0)

feature_importance = pd.DataFrame({
    'Feature': active_features,
    'Avg_Abs_Coef': avg_coef,
    'Max_Coef': np.max(logistic_coef, axis=0),
    'Min_Coef': np.min(logistic_coef, axis=0)
}).sort_values('Avg_Abs_Coef', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

zero_coef_features = feature_importance[feature_importance['Avg_Abs_Coef'] < 0.001]
small_coef_features = feature_importance[
    (feature_importance['Avg_Abs_Coef'] >= 0.001) & 
    (feature_importance['Avg_Abs_Coef'] < 0.01)
]

print(f"\nWeak Features (< 0.001): {len(zero_coef_features)}")
print(f"Small Features (0.001 - 0.01): {len(small_coef_features)}")

In [ ]:
# Feature Selection Summary
num_features = len(active_features)
eliminated = len(zero_coef_features)
retained = num_features - eliminated

print("\nFeature Selection Summary:")
print(f"Total features: {num_features}")
print(f"Features retained: {retained}")
print(f"Features eliminated: {eliminated}")
print(f"Retention rate: {retained/num_features*100:.1f}%")

In [ ]:
# Class-Specific Feature Importance
print("\nTop 10 Features by Class:")

print("\nHome Win:")
home_win_coef = pd.DataFrame({
    'Feature': active_features,
    'Coefficient': logistic_coef[2]
}).sort_values('Coefficient', key=abs, ascending=False)
print(home_win_coef.head(10).to_string(index=False))

print("\nAway Win:")
away_win_coef = pd.DataFrame({
    'Feature': active_features,
    'Coefficient': logistic_coef[1]
}).sort_values('Coefficient', key=abs, ascending=False)
print(away_win_coef.head(10).to_string(index=False))

print("\nDraw:")
draw_coef = pd.DataFrame({
    'Feature': active_features,
    'Coefficient': logistic_coef[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(draw_coef.head(10).to_string(index=False))

In [ ]:
# Confusion Matrix & Per-Class Performance
conf_matrix = cm(y_test_class, y_pred_test, labels=logistic_lasso.classes_)
confusion_df = pd.DataFrame(
    conf_matrix,
    index=[f'Actual: {c}' for c in logistic_lasso.classes_],
    columns=[f'Pred: {c}' for c in logistic_lasso.classes_]
)

print("\nConfusion Matrix:")
print(confusion_df.to_string())

print("\nPer-Class Accuracy:")
for i, class_label in enumerate(logistic_lasso.classes_):
    class_acc = conf_matrix[i, i] / conf_matrix[i].sum()
    total_samples = conf_matrix[i].sum()
    correct = conf_matrix[i, i]
    print(f"{class_label}: {class_acc*100:.2f}% ({correct}/{total_samples})")

In [ ]:
# Visualizations
import matplotlib.pyplot as plt

# Class-specific feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
class_names = ['Draw', 'Away Win', 'Home Win']

for idx, (ax, class_name, coef_row) in enumerate(zip(axes, class_names, logistic_coef)):
    coef_df = pd.DataFrame({
        'Feature': active_features,
        'Coefficient': coef_row
    }).reindex(pd.DataFrame({
        'Feature': active_features,
        'Coef_Abs': np.abs(coef_row)
    }).nlargest(15, 'Coef_Abs').index)
    
    colors = ['green' if x > 0 else 'red' for x in coef_df['Coefficient']]
    ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.7)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Coefficient')
    ax.set_title(f'{class_name}')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance by Class')
plt.tight_layout()
plt.show()

# Overall feature importance
fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(15)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_features)))
ax.barh(top_features['Feature'], top_features['Avg_Abs_Coef'], color=colors, alpha=0.8)
ax.set_xlabel('Average Absolute Coefficient')
ax.set_title('Top 15 Features Overall')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

class_counts = y_test_class.value_counts()
baseline = max(class_counts) / len(y_test_class)

print("\nModel Performance:")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Baseline Accuracy: {baseline*100:.2f}%")
print(f"Improvement: {(test_accuracy - baseline)*100:+.2f}%")

In [ ]:
# Upgraded model training switching to SVM
# Base performance with a linear hard coded SVM
from sklearn import svm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
exclude_cols = [
    'league_id', 'season', 'date', 
    'home_team_api_id', 'away_team_api_id',
    'home_team_goal', 'away_team_goal',
    'season_end', 'match_outcome'  # Exclude match_outcome from features
]

# Get feature columns
feature_cols = [col for col in df_model1.columns if col not in exclude_cols]

df_model1 = df_model1.sort_values('date').reset_index(drop=True)

X = df_model1[feature_cols]
X_scaled = scaler.fit_transform(X)
y_match = df_model1['match_outcome']

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_scaled, y_match, test_size=0.2, random_state=42
)
model = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train_class, y_train_class)

predictions = model.predict(X_test_class)
print(f"Accuracy: {accuracy_score(y_test_class, predictions)}")

Accuracy: 0.46710526315789475


In [ ]:
''' --------------------------*SKIP THIS CELL*-----------------------'''

# Grid Search for hyperparemeters
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 5, 10, 100],
    'gamma': [10, 5, 1, 0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=2)
grid.fit(X_train_class, y_train_class)

print(f"Best Parameters: {grid.best_params_}")

In [37]:
model = svm.SVC(kernel='rbf', C=100.0, gamma= 0.001)
model.fit(X_train_class, y_train_class)

predictions = model.predict(X_test_class)
print(f"Accuracy: {accuracy_score(y_test_class, predictions)}")

Accuracy: 0.49122807017543857


In [38]:
# Rolling form function needed to be called to provide more useful information

# Copy of the model incase bad things happen
rolling_df = df_model1.copy()
rolling_df = rolling_df.sort_values('date').reset_index(drop=True)

rolling_df['match_outcome'] = rolling_df.apply(
    lambda row: 'home_win' if row['home_team_goal'] > row['away_team_goal']
                else ('away_win' if row['home_team_goal'] < row['away_team_goal']
                      else 'draw'),
    axis=1
)

n = 5
implied_probs = 1 / rolling_df[['B365H', 'B365D', 'B365A']]
prob_sums = implied_probs.sum(axis=1)
rolling_df[['B365H', 'B365D', 'B365A']] = implied_probs.div(prob_sums, axis=0)
team_stats = {}

home_points_avg, away_points_avg = [], []
home_goals_avg, away_goals_avg = [], []
home_conceded_avg, away_conceded_avg = [], []

for _, row in rolling_df.iterrows():
    ht, at = row['home_team_api_id'], row['away_team_api_id']

    def get_recent(team, key, fallback=0.4):
        recent = team_stats.get(team, {}).get(key, [])
        return np.mean(recent[-n:]) if recent else fallback

    home_points_avg.append(get_recent(ht, 'points', 5))
    away_points_avg.append(get_recent(at, 'points', 5))
    home_goals_avg.append(get_recent(ht, 'goals_scored', 1.5))
    away_goals_avg.append(get_recent(at, 'goals_scored', 1.2))
    home_conceded_avg.append(get_recent(ht, 'goals_conceded', 1.2))
    away_conceded_avg.append(get_recent(at, 'goals_conceded', 1.5))

    # Update stats after recording (no data leakage)
    hg, ag = row['home_team_goal'], row['away_team_goal']
    for team, scored, conceded, dif in [(ht, hg, ag, int(hg - ag)), (at, ag, hg, int(ag - hg))]:
        if team not in team_stats:
            team_stats[team] = {'points': [], 'goals_scored': [], 'goals_conceded': []}
        if not dif:
            points = 1
        else:
            points = 3 if dif > 0 else 0
        team_stats[team]['points'].append(points)
        team_stats[team]['goals_scored'].append(scored)
        team_stats[team]['goals_conceded'].append(conceded)

rolling_df['home_points_avg'] = home_points_avg
rolling_df['away_points_avg'] = away_points_avg
rolling_df['home_goals_avg'] = home_goals_avg
rolling_df['away_goals_avg'] = away_goals_avg
rolling_df['home_conceded_avg'] = home_conceded_avg
rolling_df['away_conceded_avg'] = away_conceded_avg

In [39]:
# Using the rolling form feature

# Provide the difference which is the aspect we actuatlly want to highlight
rolling_df['rating_diff'] = rolling_df['home_rating'] - rolling_df['away_rating']

exclude_cols = [
    'league_id', 'season', 'date', 
    'home_team_api_id', 'away_team_api_id',
    'home_team_goal', 'away_team_goal',
    'season_end', 'match_outcome' 
]

# Get feature columns
feature_cols = [col for col in rolling_df.columns if col not in exclude_cols]

X = rolling_df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_match = rolling_df['match_outcome']

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_scaled, y_match, test_size=0.2, random_state=42
)


model = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train_class, y_train_class)

predictions = model.predict(X_test_class)
print(f"Accuracy: {accuracy_score(y_test_class, predictions)}")

model = svm.SVC(kernel='rbf', C=100.0, gamma= 0.001)
model.fit(X_train_class, y_train_class)

predictions = model.predict(X_test_class)
print(f"Accuracy: {accuracy_score(y_test_class, predictions)}")

Accuracy: 0.49780701754385964
Accuracy: 0.48903508771929827


In [41]:
# Adding a base feature weighing based on lasso 
feature_weights = {
    # Known best features
    'B365H': 10.0,
    'B365A': 10.0,
    'B365D': 10.0,

    # New good features
    'home_points_avg':   3.5,
    'away_points_avg':   3.5,
    'home_goals_avg':  3.0,
    'away_goals_avg':  3.0,
    'home_conceded_avg': 3.0,
    'away_conceded_avg': 3.0,
    'rating_diff': 3.0,

    # Mediocre features
    'home_rating': 1.0,
    'away_rating': 1.0,

    # Less influential features
    'home_buildUpPlaySpeed':        0.5,
    'home_buildUpPlayPassing':      0.5,
    'home_chanceCreationPassing':   0.5,
    'home_chanceCreationCrossing':  0.5,
    'home_chanceCreationShooting':  0.5,
    'home_defencePressure':         0.5,
    'home_defenceAggression':       0.5,
    'home_defenceTeamWidth':        0.5,
    'away_buildUpPlaySpeed':        0.5,
    'away_buildUpPlayPassing':      0.5,
    'away_chanceCreationPassing':   0.5,
    'away_chanceCreationCrossing':  0.5,
    'away_chanceCreationShooting':  0.5,
    'away_defencePressure':         0.5,
    'away_defenceAggression':       0.5,
    'away_defenceTeamWidth':        0.5,
}

X = rolling_df[feature_cols]
y_match = rolling_df['match_outcome']

# Scale first, then apply weights
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

weight_vector = np.array([feature_weights[f] for f in feature_cols])
X_weighted = X_scaled * weight_vector  

X_train, X_test, y_train, y_test = train_test_split(
    X_weighted, y_match, test_size=0.2, random_state=42
)

model = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

Accuracy: 0.5132


In [ ]:
''' --------------------------*SKIP THIS CELL*-----------------------'''

# Grid Search for hyperparemeters

param_grid = {
    'C': [0.1, 1, 5, 10, 100],
    'gamma': [10, 5, 1, 0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=2)
grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")

In [30]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, max_depth=None, random_state=42)

rf.fit(X_train, y_train)
preds = rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")

model = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

Accuracy: 0.4715
Accuracy: 0.5088


In [45]:
# Soley Lasso result based weights
feature_weights = {
    # Known best features
    'B365H': 5.0,
    'B365A': 5.0,
    'B365D': 5.0,

    # New good features
    'home_points_avg':   2.5,
    'away_points_avg':   2.5,
    'home_goals_avg':  2.0,
    'away_goals_avg':  2.0,
    'home_conceded_avg': 2.0,
    'away_conceded_avg': 2.0,
    'rating_diff': 2.0,

    # Mediocre features
    'home_rating': .1,
    'away_rating': .1,

    # Less influential features
    'home_buildUpPlaySpeed':        0.05,
    'home_buildUpPlayPassing':      0.05,
    'home_chanceCreationPassing':   0,
    'home_chanceCreationCrossing':  0.03,
    'home_chanceCreationShooting':  0,
    'home_defencePressure':         0.025,
    'home_defenceAggression':       0.025,
    'home_defenceTeamWidth':        0,
    'away_buildUpPlaySpeed':        0.08,
    'away_buildUpPlayPassing':      0.03,
    'away_chanceCreationPassing':   0.03,
    'away_chanceCreationCrossing':  0.015,
    'away_chanceCreationShooting':  0,
    'away_defencePressure':         0.01,
    'away_defenceAggression':       0.001,
    'away_defenceTeamWidth':        0,
}

X = rolling_df[feature_cols]
y_match = rolling_df['match_outcome']

# Scale first, then apply weights
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

weight_vector = np.array([feature_weights[f] for f in feature_cols])
X_weighted = X_scaled * weight_vector  

X_train, X_test, y_train, y_test = train_test_split(
    X_weighted, y_match, test_size=0.2, random_state=42
)

model = svm.SVC(kernel='rbf', C=5.0, gamma='scale')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

Accuracy: 0.5088


In [46]:
# Home advantage feature
home_perf = df_match.groupby('home_team_api_id')['home_team_goal'].mean()
away_perf = df_match.groupby('away_team_api_id')['away_team_goal'].mean()

team_home_adv = home_perf - away_perf

rolling_df['home_team_adv'] = rolling_df['home_team_api_id'].map(team_home_adv)

feature_weights['home_team_adv'] = 5.0

weight_vector = np.array([feature_weights[f] for f in feature_cols])
X_weighted = X_scaled * weight_vector  

X_train, X_test, y_train, y_test = train_test_split(
    X_weighted, y_match, test_size=0.2, random_state=42
)

model = svm.SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

Accuracy: 0.5088


In [47]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Encode labels: home_win=2, draw=1, away_win=0
le = LabelEncoder()
y_encoded = le.fit_transform(rolling_df['match_outcome'])

X_train_encoded, X_test_encoded, y_train_encoded, y_test_encoded = train_test_split(
    X_weighted, y_encoded, test_size=0.2, random_state=42
)

# 2. Use a model that handles non-linear patterns better than SVM
xgb_model = XGBClassifier(
    n_estimators=50,       
    max_depth=3,            
    learning_rate=0.05,     
    subsample=0.7,          
    colsample_bytree=0.7,   
    gamma=1,                
    random_state=42
)

# 3. Train on the "past", test on the "future"
xgb_model.fit(X_train_encoded, y_train_encoded)
predictions = xgb_model.predict(X_test_encoded)
print(f"Accuracy: {accuracy_score(y_test_encoded, predictions):.4f}")

Accuracy: 0.5088
